# Preparing - Load library `aleph`

## Load library

In [ ]:
import os
import torch
import numpy as np
import pandas as pd
import scanpy as sc
import torch.nn as nn
import torch.nn.functional as F
from torch import optim
from torch.autograd import Variable
import torch_geometric
from torch_geometric.data import HeteroData
from torch_geometric.loader import NeighborLoader
from torch_geometric.nn import HeteroConv, SAGEConv
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from tqdm import tqdm
from sklearn.metrics.cluster import adjusted_rand_score
import random
from sklearn import metrics
import sys

In [ ]:
# Max dimension for topological features (0 for connected components, 1 for loops)
BASE_DIR = '../data'
dataset = 'QS_Diaphragm'

MAX_TOPOLOGY_DIM = 1
topo_weight_loss = 10

warmup = 1
pre_epochs = 2
fine_epochs = 2

# 1. Module Code

### 1.1 Config

In [ ]:

class Config:
    def __init__(self):
        self.save_model = False
        self.n_clusters = 11
        self.seed = 1234
        self.device = 0 if torch.cuda.is_available() else "cpu"
        
        self.HVG = 0.05
        self.sf = True
        self.log = True
        self.normal = True
        
        self.layers = '[256,64]'
        self.lr = 0.0001
        self.fine_lr = 1.0
        self.decay = 1e-4
        self.warmup = warmup
        self.epochs = pre_epochs
        self.fine_epochs = fine_epochs
        self.batch_size = 1024
        self.ns = '[2048,1024]'
        self.recon = 'zinb'
        
        # --- scGPCL Weights ---
        self.tau = 0.25
        self.alpha = 1.0
        self.granularity = '[1,1.5,2]'
        self.lam1 = 1.0
        self.lam2 = 0.05
        self.lam3 = 1.0
        
        # --- TOPOLOGY WEIGHTS & CONFIG ---
        self.lam_topo = topo_weight_loss
        self.topo_batch_size = 64
        
        self.r = 0.99
        self.tol = 0.001
        self.df_1 = 0.2
        self.df_2 = 0.2

args = Config()
device = f'cuda:{args.device}' if torch.cuda.is_available() and isinstance(args.device, int) else "cpu"

### 1.2 Layers

In [ ]:
def set_seed(seed=0):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    random.seed(seed)
    np.random.seed(seed)

# --- LAYERS ---
class MeanAct(nn.Module):
    def forward(self, x): 
        return torch.clamp(torch.exp(x), min=1e-5, max=1e6)

class DispAct(nn.Module):
    def forward(self, x): 
        return torch.clamp(F.softplus(x), min=1e-4, max=1e4)

class ZINBLoss(nn.Module):
    def __init__(self):
        super(ZINBLoss, self).__init__()
        self.eps = 1e-10

    def forward(self, x, mean, disp, pi, scale_factor=1.0):
        # Implementation of negative log-likelihood for ZINB
        mean = mean * scale_factor.unsqueeze(1)
        try:
            t1 = torch.lgamma(disp + self.eps) + torch.lgamma(x + 1.0) - torch.lgamma(x + disp + self.eps)
        except RuntimeError as e:
            print(f"Error in Shape: X - {x.shape}, Mean - {mean.shape}, Disp - {disp.shape}, Pi - {pi.shape}")
            raise e

        t2 = (disp + x) * torch.log(1.0 + (mean / (disp + self.eps))) + (x * (torch.log(disp + self.eps) - torch.log(mean + self.eps)))
        nb_loss = t1 + t2
        zinb_loss = -torch.log(pi + (1.0 - pi) * torch.exp(-nb_loss) + self.eps)
        return torch.mean(zinb_loss)

class GNN_Encoder(nn.Module):
    def __init__(self, metadata, layer_sizes):
        super(GNN_Encoder, self).__init__()
        self.convs = nn.ModuleList()
        # Using SAGEConv with PyG's lazy initialization (-1) for bipartite passing
        for out_dim in layer_sizes:
            conv = HeteroConv({
                ('cell', 'to', 'gene'): SAGEConv((-1, -1), out_dim),
                ('gene', 'to', 'cell'): SAGEConv((-1, -1), out_dim)
            }, aggr='mean')
            self.convs.append(conv)

    def forward(self, data):
        x_dict = {'cell': data['cell'].x, 'gene': data['gene'].x}
        edge_index_dict = data.edge_index_dict
        for conv in self.convs:
            x_dict = conv(x_dict, edge_index_dict)
            x_dict = {key: F.relu(x) for key, x in x_dict.items()}
        return x_dict['cell'] # Return learned cell representations

## 1.3 Preprocessing

In [ ]:
import h5py
import torch
import numpy as np
import scanpy as sc
from anndata import AnnData
from sklearn.preprocessing import LabelEncoder
from torch_geometric.data import HeteroData

class Hete_DropFeatures:
    def __init__(self, p=None):
        assert 0. < p < 1., 'Dropout probability has to be between 0 and 1'
        self.p = p

    def __call__(self, data):
        if self.p == 0.0: return data
        for node_type in data.metadata()[0]:
            drop_mask = torch.empty((data[node_type].x.size(1),), dtype=torch.float32, device=data[node_type].x.device).uniform_(0, 1) < self.p
            data[node_type].x[:, drop_mask] = 0
        return data

def read_data(name):
    # Check both potential Kaggle directories
    path = f"{BASE_DIR}/{name}.h5"
        
    print(f"Reading data from: {path}")
    
    with h5py.File(path, 'r') as inputFile:
        X = np.array(inputFile['X']).astype('float32')
        Y = np.array(inputFile['Y'])
        
        if Y.dtype != "int64":
            encoder_x = LabelEncoder()
            Y = encoder_x.fit_transform(Y)
            
    # Convert to Scanpy AnnData object (required by scGPCL's pipeline)
    adata = AnnData(X=X)
    # The fine-tuning phase expects true labels to be stored in 'Group' for evaluation
    adata.obs['Group'] = Y 
    
    return adata

def normalize(adata, HVG=0.2, filter_min_counts=True, size_factors=True, logtrans_input=True, normalize_input=True):
    if filter_min_counts:
        sc.pp.filter_genes(adata, min_counts=1)
        sc.pp.filter_cells(adata, min_counts=1)

    n = int(adata.X.shape[1] * HVG)
    hvg_gene_idx = np.argsort(adata.X.var(axis=0))[-n:]
    adata = adata[:, hvg_gene_idx]
    adata.raw = adata.copy()

    if size_factors:
        sc.pp.normalize_total(adata)
        adata.obs['size_factors'] = adata.obs.n_counts / np.median(adata.obs.n_counts)
    else:
        adata.obs['size_factors'] = 1.0

    if logtrans_input: sc.pp.log1p(adata)
    if normalize_input: sc.pp.scale(adata)
    return adata

def construct_graph(Unnormalized_featureMatrix, featureMatrix, num_cell, num_gene):
    X = torch.tensor(Unnormalized_featureMatrix)
    cells, genes = torch.nonzero(X, as_tuple=True)

    data = HeteroData()
    data['cell'].x = torch.tensor(featureMatrix, dtype=torch.float32)
    data['cell', 'to', 'gene'].edge_index = torch.stack((cells, genes))
    data['gene', 'to', 'cell'].edge_index = torch.stack((genes, cells))

    data['cell'].num_nodes = num_cell
    data['gene'].num_nodes = num_gene  
    data['cell'].n_id = torch.arange(num_cell)
    data['gene'].n_id = torch.arange(num_gene)
    return data

## 1.4 `scGPCL` class

In [ ]:
class scGPCL(nn.Module):
    # This remains unchanged from your original script!
    def __init__(self, meta_data, num_gene, n_clusters, layer_sizes=[256,64], tau=0.25, alpha=1.0, granularity=[1,1.5,2], device=0):
        super(scGPCL, self).__init__()
        self.tau, self.alpha, self.granularity, self.emb_dim, self.n_clusters, self.device = tau, alpha, granularity, layer_sizes[-1], n_clusters, device
        self.encoder = GNN_Encoder(meta_data, layer_sizes)
        decoder_sizes = layer_sizes[::-1]
        layers = []
        for in_dim, out_dim in zip(decoder_sizes[:-1], decoder_sizes[1:]):
            layers.extend([nn.Linear(in_dim, out_dim), nn.BatchNorm1d(out_dim), nn.ReLU()])
        self.decoder = nn.Sequential(*layers)
        self.mean_layer = nn.Sequential(nn.Linear(decoder_sizes[-1], num_gene), MeanAct())
        self.disp_layer = nn.Sequential(nn.Linear(decoder_sizes[-1], num_gene), DispAct())
        self.pi_layer = nn.Sequential(nn.Linear(decoder_sizes[-1], num_gene), nn.Sigmoid())
        self.mu = nn.Parameter(torch.Tensor(n_clusters, self.emb_dim))
        self.reset_parameters()

    def encode(self, x): 
        return self.encoder(x)
    
    def decode(self, z):
        hidden = self.decoder(z)
        return self.mean_layer(hidden), self.disp_layer(hidden), self.pi_layer(hidden)
    def forward(self, view):
        rep = self.encode(view)
        mean, disp, pi = self.decode(rep)
        return mean, disp, pi, rep

    def ins_contrastive_loss(self, rep1, rep2, device=None):
        batch_size = rep1.size(0)
        pos_mask = torch.eye(batch_size, dtype=torch.float32).to(device)
        neg_mask = 1 - pos_mask
        rep1, rep2 = F.normalize(rep1, dim=1), F.normalize(rep2, dim=1)
        contrast_feature = torch.concat((rep1, rep2), dim=0)
        
        anchor_dot_contrast = torch.div(torch.matmul(contrast_feature, contrast_feature.T), self.tau)
        logits_max, _ = torch.max(anchor_dot_contrast, dim=1, keepdim=True)
        logits = anchor_dot_contrast - logits_max.detach()
        
        pos_mask, neg_mask = pos_mask.repeat(2, 2), neg_mask.repeat(2, 2)
        logits_mask = torch.scatter(torch.ones_like(pos_mask), 1, torch.arange(batch_size).view(-1, 1).to(device), 0)
        pos_mask = pos_mask * logits_mask
        
        exp_logits = torch.exp(logits) * logits_mask * (neg_mask + pos_mask)
        log_prob = logits - torch.log(exp_logits.sum(1, keepdim=True))
        
        mean_log_prob_pos = (pos_mask * log_prob).sum(1) / pos_mask.sum(1)
        loss = -mean_log_prob_pos
        return loss.mean()

    def Proto_loss(self, c_rep, cluster_assignment, centroids, num_cell_clusters):
        def cos_sim(z1, z2): return torch.matmul(F.normalize(z1, dim=-1), F.normalize(z2, dim=-1).transpose(-1, -2))
        def cos_sim_(z1, z2): return torch.bmm(F.normalize(z2, dim=-1), F.normalize(z1, dim=-1).unsqueeze(-1)).squeeze(-1)
        def Proto_NCE(rep, positives, negatives):
            f = lambda x: torch.exp(x / self.tau)
            loss_proto = 0
            for pos, neg in zip(positives, negatives):
                pos_score, neg_score = f(cos_sim(rep, pos)), f(cos_sim_(rep, neg))
                loss_proto += -torch.log(pos_score.diag() / (neg_score.sum(1) + pos_score.diag()))
            return loss_proto / len(positives)
        
        proto_positives, proto_negatives = [], []
        for pos_proto_id, protos, num_cluster in zip(cluster_assignment, centroids, num_cell_clusters):
            proto_positives.append(protos[pos_proto_id])
            neg_proto_id = np.array([list(set(range(num_cluster)) - {p}) for p in pos_proto_id]).reshape(-1, num_cluster - 1)
            proto_negatives.append(protos[neg_proto_id])
            
        return torch.mean(Proto_NCE(c_rep, proto_positives, proto_negatives))

    def soft_assign(self, z):
        q = 1.0 / (1.0 + torch.sum((z.unsqueeze(1) - self.mu)**2, dim=2) / self.alpha)
        q = q**((self.alpha+1.0)/2.0)
        return (q.t() / torch.sum(q, dim=1)).t()

    def target_distribution(self, q):
        p = q**2 / q.sum(0)
        return (p.t() / p.sum(1)).t()

    def cluster_loss(self, p, q): return torch.mean(torch.sum(p * torch.log(p / (q + 1e-6)), dim=-1))

    def infer_num_proto(self):
        num_proto = eval(self.granularity)
        if len(num_proto) == 1: self.n_cluster_list = [self.n_clusters * num_proto[0]]
        elif len(num_proto) == 2: self.n_cluster_list = [self.n_clusters * num_proto[0], int(max(self.n_clusters * num_proto[0] + 1, np.ceil(self.n_clusters * num_proto[1])))]
        elif len(num_proto) == 3: self.n_cluster_list = [int(np.floor(self.n_clusters * num_proto[0])), int(max(np.floor(self.n_clusters * num_proto[0])+1, self.n_clusters * num_proto[1])), int(max(self.n_clusters * num_proto[1]+1, np.ceil(self.n_clusters * num_proto[2])))]

    def predict_celltype(self, rep, n_clusters=None): return KMeans(n_clusters or self.n_clusters, n_init=20).fit_predict(rep)

    def predict_full_cell_rep(self, graph, gene_embedder):
        self.eval()
        with torch.no_grad():
            graph_clone = graph.clone().to(self.device)
            graph_clone['gene'].x = gene_embedder.weight
            return self.encoder(graph_clone).cpu().numpy()

    def reset_parameters(self):
        for m in self.modules():
            if isinstance(m, (nn.Linear, torch_geometric.nn.Linear)): m.reset_parameters()

In [ ]:
class embedder:
    def __init__(self, args):
        self.args = args
        os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
        os.environ['CUDA_LAUNCH_BLOCKING'] = "1"
        os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID" 
        os.environ["CUDA_VISIBLE_DEVICES"] = str(args.device)
        self.device = f'cuda:{args.device}' if torch.cuda.is_available() else "cpu"
        torch.cuda.set_device(self.device)
        
        self.model_path = os.path.join('./weights', f'{str(self.args.recon)}_{str(self.args.name)}.pt')
        os.makedirs(os.path.dirname(self.model_path), exist_ok=True)
        
    def _init_dataset(self, seed):
        adata = read_data(self.args.name)
        self.adata = normalize(adata, HVG=self.args.HVG, size_factors=self.args.sf, logtrans_input=self.args.log, normalize_input=self.args.normal)
        self.c_g_graph = construct_graph(self.adata.raw.X, self.adata.X, self.adata.n_obs, self.adata.n_vars)

        input_idx = torch.arange(self.adata.n_obs)

        generator1 = torch.Generator()
        generator1.manual_seed(seed)
        sampler1 = torch.utils.data.RandomSampler(input_idx, generator=generator1)
        
        generator2 = torch.Generator()
        # I highly recommend adding +1 to the seed here so your two contrastive views 
        # are actually different random samples!
        generator2.manual_seed(seed + 1) 
        sampler2 = torch.utils.data.RandomSampler(input_idx, generator=generator2)

        NS = eval(self.args.ns) # e.g., [2048, 1024]
        cell_nodes = ('cell', torch.ones(self.adata.n_obs).bool())
        
        # --- REPLACED HGTLoader WITH NeighborLoader ---
        self.train_loader1 = NeighborLoader(
            self.c_g_graph, 
            num_neighbors=NS,  
            sampler=sampler1, 
            input_nodes=cell_nodes, 
            batch_size=self.args.batch_size
        )
        
        self.train_loader2 = NeighborLoader(
            self.c_g_graph, 
            num_neighbors=NS, 
            sampler=sampler2, 
            input_nodes=cell_nodes, 
            batch_size=self.args.batch_size
        )
        
        self.eval_loader = NeighborLoader(
            self.c_g_graph, 
            num_neighbors=NS, 
            shuffle=False, 
            input_nodes=cell_nodes, 
            batch_size=self.args.batch_size
        )

    def Pretrain_Evaluate_Convergence(self, epoch):
        
        flag=0
        self.model.eval()
        cell_rep = self.model.predict_full_cell_rep(self.eval_loader, self.gene_embedding)
        y_pred = self.model.predict_celltype(cell_rep)
        
        if epoch == self.args.warmup+1:
            self.old_celltype_result = y_pred

        return flag

    def Fine_Evaluate_Convergence(self):
        
        flag = 0
        num = self.adata.X.shape[0]

        delta_label = np.sum(self.y_pred != self.y_pred_last).astype(np.float32) / num
        self.y_pred_last = self.y_pred
        if delta_label < self.args.tol:
            print('delta_label ', delta_label, '< tol ', self.args.tol)
            print("Reach tolerance threshold. Stopping fine-tuning.")
            flag = 1

        return flag

## 1.5 scGPCL Trainer

In [ ]:
from sklearn.decomposition import PCA
from utils.evaluation import cluster_acc, compute_F1_score
from topology_signature import TopologySignatureLoss

class scGPCL_Trainer:
    def __init__(self, args):
        self.args = args
        os.environ['CUDA_LAUNCH_BLOCKING'] = "1"
        self.device = f'cuda:{args.device}' if torch.cuda.is_available() else "cpu"
        
        self.model_path = os.path.join('./weights', f'{str(self.args.recon)}_{str(self.args.name)}.pt')
        os.makedirs(os.path.dirname(self.model_path), exist_ok=True)
        self.old_celltype_result = []

    def _init_dataset(self, adata, base_graph):
        self.adata = adata
        self.c_g_graph = base_graph.to(self.device)
        
        # Extract target variables globally for full-batch processing
        X_raw = self.adata.raw.X
        if hasattr(X_raw, "toarray"): X_raw = X_raw.toarray()
        self.X_tensor = torch.tensor(X_raw, dtype=torch.float32).to(self.device)
        self.sf_tensor = torch.tensor(self.adata.obs['size_factors'].values if hasattr(self.adata.obs['size_factors'], 'values') else self.adata.obs['size_factors'], dtype=torch.float32).to(self.device)

    def _init_model(self, base_graph=None):
        if base_graph is None:
            base_graph = self.c_g_graph.to(self.device)

        layers = eval(self.args.layers)
        self.model = scGPCL(base_graph.metadata(), self.adata.n_vars, self.args.n_clusters, layers, 
                            self.args.tau, self.args.alpha, self.args.granularity, self.device).to(self.device)
        self.gene_embedding = nn.Embedding(self.adata.n_vars, layers[-1]).to(self.device)

        param_group = [{'params': self.model.parameters(), 'lr': self.args.lr, 'weight_decay': self.args.decay},
                       {'params': self.gene_embedding.parameters(), 'lr': self.args.lr, 'weight_decay': self.args.decay}]
        self.optimizer = optim.AdamW(param_group)        
        self.recon_loss = ZINBLoss().to(self.device)

        self.transform1 = Hete_DropFeatures(self.args.df_1)
        self.transform2 = Hete_DropFeatures(self.args.df_2)

        # TopoLoss
        self.topo_loss_fn = TopologySignatureLoss(max_dimension=MAX_TOPOLOGY_DIM, loss_type="MSE", distance_type="euclidean")

    def Pretrain_Evaluate_Convergence(self, epoch):
        self.model.eval()
        cell_rep = self.model.predict_full_cell_rep(self.c_g_graph, self.gene_embedding)
        y_pred = self.model.predict_celltype(cell_rep)
        
        if epoch == self.args.warmup + 1:
            self.old_celltype_result = y_pred
            return 0
        return 0

    def pre_train(self):
        for epoch in tqdm(range(1, self.args.epochs+1), desc="Pre-training"):
            self.model.train()
            self.optimizer.zero_grad()

            # 1. Clone the graph structure for the two views
            view1 = self.c_g_graph.clone().to(self.device)
            view2 = self.c_g_graph.clone().to(self.device)

            # 2. Assign gene features FIRST (use .clone() to protect the leaf weights from in-place masking)
            view1['gene'].x = self.gene_embedding.weight.clone()
            view2['gene'].x = self.gene_embedding.weight.clone()

            # 3. NOW apply the feature dropout transformation
            view1 = self.transform1(view1)
            view2 = self.transform2(view2)

            mean1, disp1, pi1, rep1 = self.model(view1)
            mean2, disp2, pi2, rep2 = self.model(view2)

            recon_loss = (self.recon_loss(self.X_tensor, mean1, disp1, pi1, self.sf_tensor) + 
                          self.recon_loss(self.X_tensor, mean2, disp2, pi2, self.sf_tensor)) / 2.0

            ins_loss = self.model.ins_contrastive_loss(rep1, rep2, device=self.device)

            if epoch == self.args.warmup + 1:
                self.model.infer_num_proto()

            if epoch > self.args.warmup:
                cluster_assignments, centroids_list = [], []
                cell_rep = rep2.detach().cpu().numpy()
                for n_cluster in self.model.n_cluster_list:
                    y_pred = self.model.predict_celltype(cell_rep, n_clusters=n_cluster)
                    centroids = [torch.tensor(np.mean(cell_rep[y_pred==i], axis=0)) for i in np.unique(y_pred)]
                    centroids_list.append(torch.stack(centroids, dim=0).to(self.device))
                    cluster_assignments.append(y_pred)

                proto_loss = self.model.Proto_loss(rep1, cluster_assignments, centroids_list, self.model.n_cluster_list)
                topo_loss = self.topo_loss_fn(rep1, rep2) if self.args.lam_topo > 0 else torch.tensor(0.0).to(self.device)
            else:
                proto_loss = torch.tensor(0.0).to(self.device)
                topo_loss = torch.tensor(0.0).to(self.device)
                
            loss = recon_loss + self.args.lam1 * ins_loss + self.args.lam2 * proto_loss + self.args.lam_topo * topo_loss
            
            loss.backward()
            self.optimizer.step()

            if epoch % 10 == 0:
                print(f'** epochs:{epoch} | recon : {recon_loss.item():.4f} | ins-wise : {ins_loss.item():.4f} | proto : {proto_loss.item():.4f} | topo : {topo_loss.item():.4f} | loss : {loss.item():.4f}**')

            if epoch > self.args.warmup and self.Pretrain_Evaluate_Convergence(epoch) == 1:
                break

    def fine_tune(self):
        param_group = [{'params': self.model.parameters(), 'lr': self.args.fine_lr, 'weight_decay': self.args.decay, 'rho': .95},
                       {'params': self.gene_embedding.parameters(), 'lr': self.args.fine_lr, 'weight_decay': self.args.decay, 'rho': .95}]
        self.optimizer = optim.Adadelta(param_group)

        print("Initializing cluster centers with kmeans.")
        kmeans = KMeans(self.model.n_clusters, n_init=20)
        
        # Prepare evaluation graph
        eval_graph = self.c_g_graph.clone().to(self.device)
        eval_graph['gene'].x = self.gene_embedding.weight
        
        cell_rep = self.model.predict_full_cell_rep(eval_graph, self.gene_embedding)
        self.y_pred = kmeans.fit_predict(cell_rep)
        self.y_pred_last = self.y_pred
        self.model.mu.data.copy_(torch.tensor(kmeans.cluster_centers_, dtype=torch.float32).to(self.device))

        for epoch in tqdm(range(1, self.args.fine_epochs+1), desc="Fine-tuning"):
            self.model.eval()
            latent = self.model.predict_full_cell_rep(eval_graph, self.gene_embedding)
            q = self.model.soft_assign(torch.tensor(latent).to(self.device))
            p = self.model.target_distribution(q).data

            self.y_pred = torch.argmax(q, dim=1).cpu().numpy()
            
            if 'Group' in self.adata.obs:
                true_labels = self.adata.obs['Group'].values
                ari = adjusted_rand_score(true_labels, self.y_pred)
                nmi = normalized_mutual_info_score(true_labels, self.y_pred)
                ca = cluster_acc(true_labels, self.y_pred)
                ma_f1, mi_f1 = compute_F1_score(true_labels, self.y_pred)
            else:
                ari, nmi, ca, ma_f1, mi_f1 = 0, 0, 0, 0, 0

            delta_label = np.sum(self.y_pred != self.y_pred_last).astype(np.float32) / self.adata.n_obs
            self.y_pred_last = self.y_pred
            
            if epoch > 1 and delta_label < self.args.tol:
                print(f'Reach tolerance threshold (delta_label {delta_label:.5f} < tol {self.args.tol}). Stopping training.')
                break
            
            self.model.train()
            self.optimizer.zero_grad()

            # Fix order for fine-tuning as well
            view1 = self.c_g_graph.clone().to(self.device)
            view1['gene'].x = self.gene_embedding.weight.clone()
            view1 = self.transform1(view1)
            
            mean, disp, pi, rep = self.model(view1)
            
            qbatch = self.model.soft_assign(rep)
            cluster_loss = self.model.cluster_loss(Variable(p).to(self.device), qbatch)
            recon_loss = self.recon_loss(self.X_tensor, mean, disp, pi, self.sf_tensor)

            loss = recon_loss + self.args.lam3 * cluster_loss
            loss.backward()
            self.optimizer.step()

            if epoch % 10 == 0:
                print(f'** epochs:{epoch} | recon : {recon_loss.item():.4f} | cluster : {cluster_loss.item():.4f} | nmi : {nmi:.4f} | ca : {ca:.4f} | ari : {ari:.4f} **')

        return nmi, ca, ari, ma_f1, mi_f1

    def train(self, adata = None, base_graph = None, print_final_result=True):
        if adata is None:
            print("Adata not provided, initializing dataset from scratch.")
            adata = read_data(self.args.name)
            
        adata = normalize(adata, HVG=self.args.HVG, size_factors=self.args.sf, logtrans_input=self.args.log, normalize_input=self.args.normal)

        if base_graph is None:
            print("Base graph not provided, constructing graph from adata.")
            base_graph = construct_graph(adata.raw.X, adata.X, adata.n_obs, adata.n_vars)

        self._init_dataset(adata, base_graph)
        self._init_model()
        self.pre_train()
        nmi, ca, ari, ma_f1, mi_f1 = self.fine_tune()

        if print_final_result:
            print(f'\n================================ Final Score =======================================')
            print(f"** NMI : {nmi:.4f} | CA : {ca:.4f} | ARI : {ari:.4f} | Ma-F1 : {ma_f1:.4f} | Mi-F1 : {mi_f1:.4f} **")
            print(f'====================================================================================')
        
        # ADD THIS LINE
        return nmi, ca, ari, ma_f1, mi_f1


# 2. Clustering

In [ ]:
import os
import h5py
import numpy as np
from sklearn.preprocessing import LabelEncoder

print(f"Dataset: {dataset}")

# Set the name parameter for the Config
args.name = dataset
data_path = f'{BASE_DIR}/{dataset}.h5'

print("Reading labels to determine the exact number of clusters...")
with h5py.File(data_path, 'r') as inputFile:
    Y_temp = np.array(inputFile['Y'])
    if Y_temp.dtype != "int64":
        Y_temp = LabelEncoder().fit_transform(Y_temp)
    args.n_clusters = len(set(Y_temp))
    print(f"Successfully detected and assigned n_clusters = {args.n_clusters}")

# ----------------------------------------------------
# Run 1: Without Topo Loss
# ----------------------------------------------------
# print(f"\n[{dataset}] Training WITHOUT Topo loss")
# args.lam_topo = 0
# trainer_no_topo = scGPCL_Trainer(args)
# nmi_1, ca_1, ari_1, ma_f1_1, mi_f1_1 = trainer_no_topo.train()
# print(f'\tWithout Topology: NMI= {nmi_1:.4f}, ARI= {ari_1:.4f}')
# model_1 = trainer_no_topo.model

# # ----------------------------------------------------
# # Run 2: With Topo Loss
# # ----------------------------------------------------
# print(f"\n[{dataset}] Training WITH Topo loss")
# args.lam_topo = 700 # Or whatever baseline topology weight you prefer
# trainer_topo = scGPCL_Trainer(args)
# nmi_2, ca_2, ari_2, ma_f1_2, mi_f1_2 = trainer_topo.train()

# print(f'\tWith Topology:    NMI= {nmi_2:.4f}, ARI= {ari_2:.4f}')
# model_2 = trainer_topo.model

## Frozen-model robustness: topology and clustering
Run the data/model definition and clean-training cells first; skip the old noise/retraining sweeps. 
The preparation cell trains clean models only if the required retained models are missing (contrastive-sc, scGPCL, scMMN). 
Both evaluation cells use fixed model weights and the same perturbation levels/seeds. Features are perturbed after preprocessing. 
scMMN perturbs its two smoothed views; scGPCL perturbs cell features or paired cell–gene edges. 
KMeans is refit on each embedding; metrics are compared with ground truth. H0 costs use finite diagrams and diagonal matching.
Results are written to results/robustness, one topology CSV and one clustering CSV per model family and dataset.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import torch

ROBUSTNESS_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
                       if (p / "utils" / "robustness_topology.py").is_file())
if str(ROBUSTNESS_ROOT) not in sys.path:
    sys.path.insert(0, str(ROBUSTNESS_ROOT))
from utils.robustness_topology import (
    DEFAULT_LEVELS, DEFAULT_SEEDS, make_default_appliers, run_robustness_comparison,
)
from utils.robustness_clustering import run_clustering_robustness_comparison

ROBUSTNESS_LEVELS = DEFAULT_LEVELS
ROBUSTNESS_SEEDS = DEFAULT_SEEDS
# For a quick smoke run: ROBUSTNESS_LEVELS = (0.0, 0.1); ROBUSTNESS_SEEDS = (33333,)
ROBUSTNESS_OUTPUT_DIR = ROBUSTNESS_ROOT / "results" / "robustness"

def _require_trained(*names):
    missing = [name for name in names if name not in globals()]
    if missing:
        raise RuntimeError("Run the clean-model training cell first: " + ", ".join(missing))

def _torch_encoder(model, select_instance=False):
    def encode(x, aux):
        model.eval()
        model_device = next(model.parameters()).device
        x = torch.as_tensor(x, dtype=torch.float32, device=model_device)
        with torch.no_grad():
            z = model(x, None)[0] if select_instance else model(x)
        return z.detach().cpu()
    return encode
from copy import deepcopy

# Retain trainers: inference requires both the cell encoder and learned gene embeddings.
if "trainer_no_topo" not in globals() or "trainer_topo" not in globals():
    args_base, args_topo = deepcopy(args), deepcopy(args)
    args_base.lam_topo, args_topo.lam_topo = 0, TOPO_WEIGHT
    trainer_no_topo = scGPCL_Trainer(args_base)
    trainer_no_topo.train()
    trainer_topo = scGPCL_Trainer(args_topo)
    trainer_topo.train()
if not np.array_equal(trainer_no_topo.adata.obs_names, trainer_topo.adata.obs_names):
    raise ValueError("Clean trainers must preserve the same cell ordering.")
if not np.array_equal(trainer_no_topo.adata.var_names, trainer_topo.adata.var_names):
    raise ValueError("Clean trainers must use the same feature ordering.")

def _hetero_encoder(trainer):
    def encode(x, graph):
        trainer.model.eval()
        trainer.gene_embedding.eval()
        graph = graph.clone().to(trainer.device)
        graph['cell'].x = torch.as_tensor(x, dtype=torch.float32, device=trainer.device)
        with torch.no_grad():
            return trainer.model.predict_full_cell_rep(graph, trainer.gene_embedding)
    return encode

def _hetero_edge_dropout(x, graph, level, seed):
    graph = graph.clone()
    # Cell and gene indices belong to DIFFERENT node sets: sample directed pairs.
    edge_applier = make_default_appliers(["edge_dropout"], force_undirected=False)["edge_dropout"]
    x, edges = edge_applier(x, graph['cell', 'to', 'gene'].edge_index, level, seed)
    graph['cell', 'to', 'gene'].edge_index = edges
    graph['gene', 'to', 'cell'].edge_index = edges.flip(0)
    return x, graph

pert_list = ["feature_dropout", "gaussian_noise", "edge_dropout"]
ROBUSTNESS_APPLIERS = make_default_appliers(["feature_dropout", "gaussian_noise"])
ROBUSTNESS_APPLIERS["edge_dropout"] = _hetero_edge_dropout
models_for_robustness = {
    label: dict(get_representation_fn=_hetero_encoder(trainer),
                X_clean=trainer.c_g_graph['cell'].x.detach().clone(),
                aux_clean=trainer.c_g_graph.clone(), perturbation_appliers=ROBUSTNESS_APPLIERS)
    for label, trainer in [("Without Topology", trainer_no_topo), ("With Topology", trainer_topo)]
}
# Use post-filtering labels, not the original unfiltered H5 rows.
robustness_labels = np.asarray(trainer_no_topo.adata.obs['Group']).reshape(-1)
if not np.array_equal(robustness_labels, np.asarray(trainer_topo.adata.obs['Group']).reshape(-1)):
    raise ValueError("Clean trainers have different label orderings.")

robustness_num_clusters = len(np.unique(robustness_labels))
robustness_dataset = str(globals().get("DATASET", globals().get("dataset", "dataset")))
robustness_file_prefix = "scgpcl" + "_" + Path(robustness_dataset).name


In [ ]:
topology_results = run_robustness_comparison(
    models=models_for_robustness, dataset_name=robustness_dataset,
    perturbations=pert_list, levels=ROBUSTNESS_LEVELS, seeds=ROBUSTNESS_SEEDS,
    save_path=ROBUSTNESS_OUTPUT_DIR / (robustness_file_prefix + "_topology.csv"),
)
display(topology_results.head())


In [ ]:
clustering_results = run_clustering_robustness_comparison(
    models=models_for_robustness, dataset_name=robustness_dataset,
    Y_true=robustness_labels, num_clusters=robustness_num_clusters,
    perturbations=pert_list, levels=ROBUSTNESS_LEVELS, seeds=ROBUSTNESS_SEEDS,
    save_path=ROBUSTNESS_OUTPUT_DIR / (robustness_file_prefix + "_clustering.csv"),
)
display(clustering_results.head())
